# MIPO-NDD: resume the DGX mechanism run on Colab

Continues `v4_validation_mechanism_650M_bafe6e1c78fd218e` (4 models x 3 seeds x 10 folds = 120 runs).
The backup holds mipo seed 42 folds 0-6 finished and fold 7 saved after epoch 2.
Fold 7 was started on the DGX's torch/Python, so on Colab it restarts from epoch 0; the partial run is kept
in `set_aside_partial_runs/`, not deleted.

1. Upload `MIPO_NDD_colab_resume.zip` (or the original `MIPO_NDD_backup.zip`) to the top level of **My Drive**.
   Keep your local copy until the run is finished. See `README_COLAB_RESUME.md` at the top of the ZIP.
2. Runtime -> Change runtime type -> GPU (A100 or H100; High-RAM if offered).
3. Run cell 1 once. It restores into `MyDrive/MIPO_NDD` and never overwrites a file.
   An existing release ZIP with different contents is renamed and kept.
4. Run cell 2 to train. It runs up to `PARALLEL = 4` model/seed/fold jobs at once on the one GPU,
   starting them 8 minutes apart and only while RAM allows. Each job logs to `logs/` in the experiment
   folder; the cell prints a status line every 10 minutes.
   After a disconnect, run cell 2 again: finished runs are skipped, and interrupted runs resume from their
   last completed epoch (on the same torch/Python; after a Colab image change they are set aside and restarted).
   Cell 2 stops before training if the protocol hash is not the DGX one, or if another runtime is training.
5. Cell 3 shows progress at any time without training.


In [ ]:
# 1. RESTORE the DGX backup into MyDrive/MIPO_NDD. Never overwrites; skips itself once done.
BACKUP_NAMES = ('MIPO_NDD_colab_resume.zip', 'MIPO_NDD_backup.zip')  # new bundle first, original backup also works
EXPECTED_RELEASE = '3ce6c2822684b45fde1a7cbe4505980c7b2fbf5736eeead0e69eb1f1bad9917a'
EXPERIMENT_NAME = 'v4_validation_mechanism_650M_bafe6e1c78fd218e'
RESTORE = (f'MIPO_NDD/{EXPERIMENT_NAME}/',
           'MIPO_NDD/v4_validation_mechanism_650M_2db28568d92f6b28/',  # AMP-collapse diagnostics, kept for the record
           'MIPO_NDD/v4_650M_masked/',
           'MIPO_NDD/mipo_ndd_v4_colab_upload.zip')
RELEASE = 'MIPO_NDD/mipo_ndd_v4_colab_upload.zip'

import hashlib, json, os, shutil, time, zipfile, zlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass
MYDRIVE = Path(os.environ.get('MIPO_MYDRIVE', '/content/drive/MyDrive'))
ROOT = MYDRIVE/'MIPO_NDD'
MARKER = ROOT/f'.restored_{EXPERIMENT_NAME}'

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def crc(path):
    value = 0
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            value = zlib.crc32(chunk, value)
    return value

if MARKER.exists():
    print('Already restored. Training progress on Drive is newer than the backup, so nothing was touched.')
else:
    backup = next((folder/name for name in BACKUP_NAMES for folder in (MYDRIVE, ROOT) if (folder/name).exists()), None)
    assert backup, f'Upload {BACKUP_NAMES[0]} to My Drive first'
    local = Path(os.environ.get('MIPO_LOCAL', '/content'))/backup.name
    if not local.exists() or local.stat().st_size != backup.stat().st_size:
        print('Copying the backup from Drive to local disk ...', flush=True)
        shutil.copyfile(backup, local)
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None, 'Backup ZIP is corrupt; upload it again'
        members = [i for i in z.infolist() if not i.is_dir() and i.filename.startswith(RESTORE)]
        todo, conflicts = [], []
        for i in members:
            target = MYDRIVE/i.filename
            if not target.exists():
                todo.append(i)
            elif target.stat().st_size != i.file_size or crc(target) != i.CRC:
                conflicts.append(i.filename)
        release_conflict = RELEASE in conflicts
        if release_conflict:
            conflicts.remove(RELEASE)
        assert not conflicts, (f'{len(conflicts)} files on Drive differ from the backup; nothing was changed. '
                               f'Move them aside first: {conflicts[:5]}')
        if release_conflict:
            old = MYDRIVE/RELEASE
            kept = old.with_name(f'mipo_ndd_v4_colab_upload_old_{sha(old)[:12]}.zip')
            old.rename(kept)
            print('A different release ZIP was already on Drive; kept as', kept.name)
            todo.append(z.getinfo(RELEASE))
        print(f'{len(members)} backup files: {len(members)-len(todo)} already on Drive, extracting {len(todo)} ...', flush=True)
        for n, i in enumerate(todo, 1):
            target = MYDRIVE/i.filename
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_name(target.name+'.partial')
            with z.open(i) as source, open(partial, 'wb') as sink:
                shutil.copyfileobj(source, sink, 1 << 20)
            partial.replace(target)
            stamp = time.mktime(i.date_time+(0, 0, -1))
            os.utime(target, (stamp, stamp))
            if n % 250 == 0 or n == len(todo):
                print(f'  {n}/{len(todo)}', flush=True)
    assert sha(ROOT/'mipo_ndd_v4_colab_upload.zip') == EXPECTED_RELEASE, 'Release ZIP on Drive is not the DGX release'
    MARKER.write_text(f'Restored from {backup}\n')
    print('Restore complete:', ROOT/EXPERIMENT_NAME)


Mounted at /content/drive
Already restored. Training progress on Drive is newer than the backup, so nothing was touched.


In [ ]:

STORAGE = 'drive'                      # Runs live on Drive so a disconnect loses at most one epoch
SCOPE = 'mechanism'                    # mechanism (10 genes), direct, or curriculum
MODEL_SIZE = '650M'
USE_LLR = True
TEST_GENES = None                      # None: fixed 10-gene mechanism panel; direct/curriculum: 3-gene pilot
SEEDS = [42, 123, 2026]                # Training variability; seeds are not independent biological samples
SPLIT_SEED = 42                        # Same validation/calibration split across model seeds.
VALIDATION_GROUPS = 5
CALIBRATION_GROUPS = 3
STEPS_PER_EPOCH = 3000
EPOCHS = 20                            # Direct mode only; curriculum uses its stage budgets.
PATIENCE = 6
DO_FEATURES = False                    # Restored cache is complete; False never reloads ESM or rewrites the cache
DO_TRAIN = True
DO_BASELINE = True
DO_ABLATIONS = True
ABLATIONS = 'esm_mlp,no_structure,no_global_kernel'  # Matched component controls; one seed is a pilot only
EXPECTED_EXPERIMENT = 'v4_validation_mechanism_650M_bafe6e1c78fd218e'  # DGX run; None allows a new protocol folder
SEED_FIRST = True                      # Finish all 4 models for seed 42 before seed 123 (order only; same results)
PARALLEL = 10                          # Training processes sharing the GPU; 1 = one at a time. Same results either way

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'], check=True)
    import pandas as pd
from IPython.display import display

# Colab provides /content; on a plain JupyterLab host (e.g. DGX) use the notebook's directory.
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()

assert STORAGE in {'local', 'drive'}
if STORAGE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_ROOT = Path('/content/drive/MyDrive/MIPO_NDD')
else:
    STORAGE_ROOT = WORK_ROOT
ARCHIVE = STORAGE_ROOT/'mipo_ndd_v4_colab_upload.zip'
assert ARCHIVE.exists(), f'Upload the updated release ZIP to {ARCHIVE}'

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def banner(text):
    print('\n'+'='*78+'\n'+text+'\n'+'='*78, flush=True)

def run_script(script, *args):
    subprocess.run([sys.executable, str(PROJECT/'scripts'/script), *map(str, args)], check=True)

def run(*args):
    subprocess.run([sys.executable, '-m', 'mipo', *map(str, args)], check=True)

banner('1. INSTALL UPDATED RELEASE')
archive_hash = sha(ARCHIVE)
PROJECT = WORK_ROOT/f'mipo_ndd_v4_validation_{archive_hash[:12]}'
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    for item in archive.infolist():
        target = (PROJECT/item.filename).resolve()
        if target != PROJECT and PROJECT not in target.parents:
            raise ValueError('Unsafe archive path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]'], check=True)
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
print('Release SHA256:', archive_hash)
assert SCOPE in {'mechanism', 'direct', 'curriculum'}
if TEST_GENES is None:
    TEST_GENES = (pd.read_csv(PROJECT/'v4/robust/mechanism_panel.csv').gene.tolist()
                  if SCOPE == 'mechanism' else ['ASPA', 'KCNQ2', 'PAX6'])
assert MODEL_SIZE in {'150M', '650M'}
assert SEEDS and TEST_GENES and len(set(SEEDS)) == len(SEEDS) and len(set(TEST_GENES)) == len(TEST_GENES)
assert VALIDATION_GROUPS >= 2 and CALIBRATION_GROUPS >= 1
assert STEPS_PER_EPOCH > 0 and EPOCHS > 0 and PATIENCE > 0

banner('2. FIX SPLITS AND CHECKPOINT SELECTION BEFORE TRAINING')
DATA_SCOPE = 'curriculum' if SCOPE == 'mechanism' else SCOPE
CORPUS = PROJECT/f'v4/robust/{DATA_SCOPE}.csv.gz'
RESOURCES = PROJECT/'v4/resources'
METADATA = PROJECT/'v4/robust/assay_metadata.csv'
SOURCE_SPLITS = PROJECT/f'v4/robust/{SCOPE}_splits.json'
print('Primary analysis excludes consensus and documented duplicate deposits; see v4/robust/build_report.json')
SPLITS = PROJECT/f'active_{SCOPE}_multigene_splits.json'
run_script('prepare_validation_splits.py', '--corpus', CORPUS, '--source', SOURCE_SPLITS,
           '--out', SPLITS, '--validation-groups', VALIDATION_GROUPS,
           '--calibration-groups', CALIBRATION_GROUPS, '--seed', SPLIT_SEED,
           '--report-test-genes', ','.join(TEST_GENES), '--resources', RESOURCES)
split = json.loads(SPLITS.read_text())
support = json.loads(SPLITS.with_suffix('.support.json').read_text())
identity = json.loads(SPLITS.with_suffix('.identity.json').read_text())
if identity['flagged']:
    print(f"WARNING: test genes within {identity['flag_threshold']:.0%} sequence identity of a train gene "
          f"(similarity diagnostic; assess homology-aware splits before interpreting generalization): {identity['flagged']}")
folds = sorted({i for gene in TEST_GENES for i, fold in enumerate(split['folds']) if gene in fold['test']})
found = {g for i in folds for g in split['folds'][i]['test']}
assert set(TEST_GENES) <= found, f'Test genes absent from outer folds: {set(TEST_GENES)-found}'
for i in folds:
    fold = split['folds'][i]
    print(f"Fold {i}: test={fold['test']}, validation={fold['validation']}, calibration={fold['calibration']}, train genes={len(fold['train'])}")
    counts = support[str(fold['fold'])]
    print('Number of genes per task in each role:')
    display(pd.DataFrame(counts).T)
    for task, roles in counts.items():
        if roles['test'] and (not roles['validation'] or not roles['calibration']):
            print(f'Task coverage gap: {task}. Validation/calibration transfer for this task is not established.')
# Joint training for the mixed NDD/non-NDD mechanism panel; NDD specialization would
# change the target population. The measured broad corpus remains gene-held-out.
CONFIG_SCOPE = 'direct' if SCOPE == 'mechanism' else SCOPE
config = json.loads((PROJECT/f'configs/v4_robust_{CONFIG_SCOPE}.json').read_text())
config.update(seed=SEEDS[0], use_llr=USE_LLR, require_llr=USE_LLR,
              steps_per_epoch=STEPS_PER_EPOCH, patience=PATIENCE,
              selection_metric='macro_gene_spearman', min_validation_genes=VALIDATION_GROUPS)
config['amp'] = False
if 'stages' in config:
    print('Curriculum stage budgets:', [(s['name'], s['epochs']) for s in config['stages']])
    max_epochs = sum(s['epochs'] for s in config['stages'])
else:
    config['epochs'] = EPOCHS
    max_epochs = EPOCHS
CONFIG = PROJECT/'active_v4_validation.json'
CONFIG.write_text(json.dumps(config, indent=2))
models = list(dict.fromkeys((['mipo'] if DO_TRAIN else []) +
                           ([x.strip() for x in ABLATIONS.split(',') if x.strip()] if DO_ABLATIONS else [])))
protocol = {'release_sha256': archive_hash, 'corpus_sha256': sha(CORPUS),
            'split_sha256': sha(SPLITS), 'metadata_sha256': sha(METADATA),
            'config': config, 'model_size': MODEL_SIZE, 'seeds': SEEDS, 'folds': folds, 'models': models, 'release_policy': 'robust_measured_v1'}
protocol_hash = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
EXPERIMENT = STORAGE_ROOT/f'v4_validation_{SCOPE}_{MODEL_SIZE}_{protocol_hash}'
if EXPECTED_EXPERIMENT and EXPERIMENT.name != EXPECTED_EXPERIMENT:
    raise RuntimeError(f'Protocol changed: this would train into {EXPERIMENT.name}, not {EXPECTED_EXPERIMENT}. '
                       'Check the release ZIP and the SEEDS/ABLATIONS/amp settings. Nothing was trained.')
EXPERIMENT.mkdir(parents=True, exist_ok=True)
for source, name in [(CONFIG, 'config.json'), (SPLITS, 'splits.json')]:
    shutil.copy2(source, EXPERIMENT/name)
(EXPERIMENT/'protocol.json').write_text(json.dumps(protocol, indent=2))
RUNS = EXPERIMENT/'runs'
print('Outputs:', EXPERIMENT)
print(f'{len(models)*len(SEEDS)*len(folds)} requested model/seed/fold runs: {models}')
print(f"{STEPS_PER_EPOCH} batches x {config['batch_size']} = {STEPS_PER_EPOCH*config['batch_size']:,} sample draws/epoch; max {max_epochs} epochs; patience {PATIENCE}")
print('Selection: mean assay Spearman within each validation gene, then equal mean across genes.')
print('Expanded holdouts change training data; compare models within this new protocol.')

banner('3. REUSE OR EXTRACT FROZEN FEATURES')
MODEL = {'150M': 'facebook/esm2_t30_150M_UR50D', '650M': 'facebook/esm2_t33_650M_UR50D'}[MODEL_SIZE]
FEATURE_TAG = f'v4_{MODEL_SIZE}_{"masked" if USE_LLR else "no_llr"}'
CACHE = STORAGE_ROOT/FEATURE_TAG
LOCAL_CACHE = CACHE if STORAGE == 'local' else PROJECT/FEATURE_TAG
FEATURE_ARCHIVE = STORAGE_ROOT/f'{FEATURE_TAG}.zip'
if not (CACHE/'manifest.json').exists() and FEATURE_ARCHIVE.exists():
    with zipfile.ZipFile(FEATURE_ARCHIVE) as archive:
        names = set(archive.namelist())
        if f'{FEATURE_TAG}/manifest.json' in names:
            destination = STORAGE_ROOT
        elif 'manifest.json' in names:
            destination = CACHE
        else:
            raise ValueError(f'{FEATURE_ARCHIVE} must contain manifest.json at its root or inside {FEATURE_TAG}/')
        destination.mkdir(parents=True, exist_ok=True)
        destination = destination.resolve()
        for item in archive.infolist():
            target = (destination/item.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError('Unsafe feature archive path')
        archive.extractall(destination)
    print('Uploaded feature cache extracted:', CACHE)
# Reuse embeddings but fill newly measured sites when an existing cache is supplied.
extract_features = DO_FEATURES is not False
if extract_features:
    print('Building/resuming ESM embeddings and masked LLR. This is a separate GPU stage before training.')
    args = ['features', '--corpus', CORPUS, '--resources', RESOURCES, '--cache', CACHE, '--model', MODEL]
    if not USE_LLR:
        args.append('--no-llr')
    run(*args)
assert (CACHE/'manifest.json').exists(), f'Feature cache missing: {CACHE}; set DO_FEATURES=True'
if CACHE.resolve() != LOCAL_CACHE.resolve():
    shutil.copytree(CACHE, LOCAL_CACHE, dirs_exist_ok=True)
print('Local cache ready:', LOCAL_CACHE)

banner('4. TRAIN AND RUN MATCHED ABLATIONS')
if models:
    # train.py resumes a partial run only on the torch/Python that started it. A partial run from another
    # environment (e.g. the DGX's fold 7) is moved out of runs/, kept, and that fold restarts here.
    runtime = subprocess.check_output([sys.executable, '-c', 'import platform, torch; '
                                       'print(torch.__version__, platform.python_version())'], text=True).split()
    for last in sorted(RUNS.glob('*/seed_*/fold_*/last.pt')):
        partial = last.parent
        if (partial/'test_summary.json').exists():
            continue
        saved = torch.load(last, map_location='cpu', weights_only=True)['provenance']
        if [saved['torch'], saved['python']] != runtime:
            target = (EXPERIMENT/'set_aside_partial_runs'/partial.relative_to(RUNS).parent/
                      f"{partial.name}_{time.strftime('%Y%m%d-%H%M%S')}")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(partial), str(target))
            print(f"Set aside partial run from torch {saved['torch']} / Python {saved['python']}; it restarts on "
                  f"torch {runtime[0]} / Python {runtime[1]}. Kept at {target.relative_to(EXPERIMENT)}", flush=True)

    # One process per model/seed/fold; each writes only its own run folder and its log in EXPERIMENT/logs.
    # A job starts when a slot is free, the previous start was STAGGER_S ago, and free RAM is at least
    # 1.5x the largest job measured so far plus 2 GB. One job always runs, whatever the RAM.
    # A run whose predictions collapse to a constant at epoch 0 is retried once, after all other runs.
    import platform
    import psutil
    POLL_S, STAGGER_S, STATUS_S, STALE_S = 20, 120, 600, 600
    order = ([(m, s, f) for s in SEEDS for f in folds for m in models] if SEED_FIRST else
             [(m, s, f) for m in models for s in SEEDS for f in folds])

    def run_dir(job):
        return RUNS/job[0]/f'seed_{job[1]}'/f'fold_{job[2]:02d}'

    def label(job):
        return f'{job[0]}_seed{job[1]}_fold{job[2]:02d}'

    pending = [job for job in order if not (run_dir(job)/'test_summary.json').exists()]
    print(f'{len(order)-len(pending)} of {len(order)} runs finished; {len(pending)} to train, up to {PARALLEL} at a time', flush=True)
    LOGS = EXPERIMENT/'logs'
    LOGS.mkdir(exist_ok=True)
    SESSION = EXPERIMENT/'.training_session.json'
    session = f'{platform.node()}-{os.getpid()}-{int(time.time())}'
    try:
        other = json.loads(SESSION.read_text())
    except (OSError, ValueError):
        other = None
    if other and time.time()-other['heartbeat'] < STALE_S:
        raise RuntimeError(f"Runtime {other['session']} was training in this folder {(time.time()-other['heartbeat'])/60:.0f} min ago. "
                           'Two runtimes would write the same runs. Stop the other one, wait 10 minutes, then run this cell again.')
    threads = str(max(1, (os.cpu_count() or 1)//PARALLEL))
    env = {**os.environ, 'OMP_NUM_THREADS': threads, 'MKL_NUM_THREADS': threads, 'PYTHONUNBUFFERED': '1'}
    running, peak_gb, kills, failed = {}, {}, {}, []
    collapses, deferred = {}, []
    slots, finished, last_launch, last_status = PARALLEL, 0, 0., 0.

    def rss_gb(proc):
        try:
            parent = psutil.Process(proc.pid)
            return sum(p.memory_info().rss for p in [parent, *parent.children(recursive=True)])/1024**3
        except psutil.Error:
            return 0.

    shown = {}

    def epoch_records(job):
        # A job may be mid-way through writing a line; skip incomplete lines instead of crashing.
        path = LOGS/f'{label(job)}.log'
        records = []
        for line in (path.read_text(errors='replace').splitlines() if path.exists() else []):
            if line.startswith('{"epoch"'):
                try:
                    records.append(json.loads(line))
                except ValueError:
                    continue
        return records

    def last_epoch(job):
        records = epoch_records(job)
        return records[-1]['epoch'] if records else '-'

    def print_new_epochs(job):
        records = epoch_records(job)
        for r in records[shown.get(job, len(records)):]:
            note = 'best so far' if r['improved'] else f"no improvement {r['bad_epochs']}/{PATIENCE}"
            print(f"{time.strftime('%H:%M')} {label(job)} epoch {r['epoch']}: loss {r['loss']:.3f}, "
                  f"val spearman {r['validation_spearman']:.3f} ({note})", flush=True)
        shown[job] = len(records)

    def gpu_use():
        try:
            return subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader'],
                                  capture_output=True, text=True).stdout.strip()
        except OSError:
            return 'n/a'

    started = time.time()
    try:
        while pending or running or deferred:
            for job, (proc, log) in list(running.items()):
                peak_gb[job] = max(peak_gb.get(job, 0.), rss_gb(proc))
                print_new_epochs(job)

                code = proc.poll()
                if code is None:
                    continue
                log.close()
                del running[job]
                if code == 0:
                    finished += 1
                    print(f"{time.strftime('%H:%M')} finished {label(job)}", flush=True)
                elif code == -9 and kills.get(job, 0) < 2:
                    kills[job] = kills.get(job, 0)+1
                    slots = max(1, slots-1)
                    pending.insert(0, job)
                    print(f"{time.strftime('%H:%M')} {label(job)} was killed by the system, usually for lack of RAM. "
                          f"Requeued to resume from its last epoch; now at most {slots} at a time", flush=True)
                else:
                    text = (LOGS/(label(job)+'.log')).read_text(errors='replace').rsplit('\n=== ', 1)[-1]
                    if 'Validation Spearman undefined' in text and not collapses.get(job):
                        collapses[job] = 1
                        deferred.append(job)
                        print(f"{time.strftime('%H:%M')} {label(job)} collapsed to constant predictions at epoch 0; "
                              "retrying once after the other runs", flush=True)
                    else:
                        failed.append(job)
                        print(f"{time.strftime('%H:%M')} FAILED {label(job)} (exit {code}); see {LOGS/(label(job)+'.log')}", flush=True)
                        if len(failed) >= 3 and not finished and pending:
                            pending.clear()
                            print('Three runs failed before any finished; no new runs will start.', flush=True)
            if not pending and deferred and len(running) < slots:
                pending.extend(deferred)
                deferred.clear()
                print(f"{time.strftime('%H:%M')} retrying {len(pending)} collapsed run(s) once", flush=True)
            free = psutil.virtual_memory().available/1024**3
            need = 1.5*max(peak_gb.values(), default=0.)+2
            if pending and (not running or (len(running) < slots and time.time()-last_launch >= STAGGER_S and free >= need)):
                job = pending.pop(0)
                log = open(LOGS/f'{label(job)}.log', 'a')
                log.write(f"\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} session {session}\n")
                log.flush()
                command = [sys.executable, PROJECT/'scripts'/'run_experiments.py', '--corpus', CORPUS,
                           '--resources', RESOURCES, '--cache', LOCAL_CACHE, '--splits', SPLITS, '--config', CONFIG,
                           '--metadata', METADATA, '--out', RUNS, '--folds', job[2], '--seeds', job[1], '--ablations', job[0]]
                running[job] = (subprocess.Popen(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT, env=env), log)
                shown[job] = len(epoch_records(job))
                last_launch = time.time()
                print(f"{time.strftime('%H:%M')} started {label(job)} ({len(running)} running, {free:.0f} GB RAM free)", flush=True)
            beat = SESSION.with_suffix('.tmp')
            beat.write_text(json.dumps({'session': session, 'heartbeat': time.time(), 'running': [label(job) for job in running]}))
            beat.replace(SESSION)
            if time.time()-last_status >= STATUS_S:
                last_status = time.time()
                done = sum((run_dir(job)/'test_summary.json').exists() for job in order)
                print(f"--- {time.strftime('%H:%M')}: {done}/{len(order)} runs finished, {len(pending)} waiting, "
                      f"{len(deferred)} collapsed and waiting for a retry, "
                      f"{free:.0f} GB RAM free (next start needs {need:.0f}), GPU {gpu_use()}", flush=True)
                for job, (proc, _) in running.items():
                    print(f'    {label(job):34s} last finished epoch {last_epoch(job)}, RAM {rss_gb(proc):.1f} GB', flush=True)
            time.sleep(POLL_S)
    finally:
        if running:
            print('Stopping running jobs; each resumes from its last finished epoch next time.', flush=True)
        for proc, _ in running.values():
            proc.terminate()
        for proc, log in running.values():
            try:
                proc.wait(60)
            except subprocess.TimeoutExpired:
                proc.kill()
            log.close()
        try:
            if json.loads(SESSION.read_text())['session'] == session:
                SESSION.unlink()
        except (OSError, ValueError):
            pass
    print(f'Run time: {(time.time()-started)/60:.1f} min')
    if collapses:
        recovered = [label(job) for job in collapses if job not in failed]
        print(f'Collapsed at epoch 0 in this session: {[label(job) for job in collapses]}. '
              f'Recovered on retry: {recovered or "none"}.')
    if failed:
        raise RuntimeError(f'{len(failed)} runs failed: {[label(job) for job in failed]}. Their logs are in {LOGS}. '
                           'Run this cell again to retry them; each resumes from its last finished epoch.')

banner('5. TRAINING HISTORY AND VALIDATION DIAGNOSTICS')
selected_runs = []
for path in sorted(RUNS.glob('*/seed_*/fold_*/history.json')):
    if int(path.parent.name.split('_')[1]) not in folds or int(path.parent.parent.name.split('_')[1]) not in SEEDS:
        continue
    selected_runs.append(path.parent)
    print('\n', path.parent.relative_to(RUNS))
    history = pd.DataFrame(json.loads(path.read_text()))
    losses = pd.json_normalize(history.loss_components).add_prefix('train_')
    display(pd.concat([history[['epoch', 'loss', 'validation_spearman', 'lr', 'bad_epochs']], losses], axis=1).round(5))
    status = json.loads((path.parent/'training_status.json').read_text())
    print('Saved checkpoint / stopping:', status)
    per_gene = pd.DataFrame(history.validation_per_gene.tolist(), index=history.epoch)
    display(per_gene.round(4))
    print('Best checkpoint validation by gene and assay:')
    validation = pd.read_csv(path.parent/'validation_metrics.csv')
    display(validation[['gene', 'task', 'n', 'spearman', 'bias', 'mean_sigma', 'coverage_90_gaussian']].round(4))
    print('The history describes this validation panel; it does not establish the cause of a plateau.')

if DO_BASELINE and USE_LLR:
    banner('6. EXACT-ROW CACHED LLR BASELINE')
    run_script('compare_cached_llr.py', '--corpus', CORPUS, '--resources', RESOURCES,
               '--cache', LOCAL_CACHE, '--runs', RUNS)

banner('7. TEST RANKING AND INTERVAL DIAGNOSTICS')
if SCOPE == 'mechanism' and selected_runs:
    run_script('evaluate_property_contrasts.py', '--runs', RUNS,
               '--out', EXPERIMENT/'paired_property_contrasts.csv')
    print('Paired rank contrasts are descriptive assay-separation diagnostics, not causal folding labels.')
comparisons, intervals = [], []
for folder in selected_runs:
    model = folder.parents[1].name
    seed = int(folder.parent.name.split('_')[1])
    fold = int(folder.name.split('_')[1])
    metrics_path = folder/'test_metrics.csv'
    if metrics_path.exists():
        metrics = pd.read_csv(metrics_path)
        metrics.insert(0, 'model', model)
        metrics.insert(1, 'seed', seed)
        metrics.insert(2, 'fold', fold)
        intervals.append(metrics)
    comparison = folder/'matched_llr_metrics.csv'
    if comparison.exists():
        rows = pd.read_csv(comparison)
        rows.insert(0, 'model', model)
        rows.insert(1, 'seed', seed)
        rows.insert(2, 'fold', fold)
        comparisons.append(rows)
if comparisons:
    table = pd.concat(comparisons, ignore_index=True)
    if SCOPE == 'mechanism':
        panel = pd.read_csv(PROJECT/'v4/robust/mechanism_panel.csv').set_index('gene')
        table['evaluation_part'] = table.gene.map(panel.part)
        table['previously_examined'] = table.gene.map(panel.previously_examined)
    else:
        table['evaluation_part'] = 'ndd_evaluation'
    table.to_csv(EXPERIMENT/'matched_comparison.csv', index=False)
    display(table.round(4))
    if table[['model_spearman', 'llr_spearman']].notna().all().all():
        gene_scores = table.groupby(['model', 'seed', 'evaluation_part', 'gene'])[['model_spearman', 'llr_spearman', 'delta']].mean()
        per_seed = gene_scores.groupby(['model', 'seed', 'evaluation_part']).mean()
        print('Macro across genes, separately for each seed and NDD/non-NDD panel (all assays retained):')
        display(per_seed.round(4))
        per_seed.to_csv(EXPERIMENT/'macro_comparison_by_seed.csv')
        if len(SEEDS) > 1:
            print('Mean and standard deviation across seeds; this is training variability, not uncertainty across genes:')
            display(per_seed.groupby(['model', 'evaluation_part']).agg(['mean', 'std']).round(4))
    else:
        print('Some correlations are undefined: inspect the assay table; headline macro comparison suppressed.')
if intervals:
    table = pd.concat(intervals, ignore_index=True)
    table.to_csv(EXPERIMENT/'test_diagnostics.csv', index=False)
    display(table[['model', 'seed', 'gene', 'task', 'n', 'spearman', 'bias', 'mean_sigma',
                   'coverage_90_gaussian', 'coverage_calibrated', 'calibrated_interval_rows']].round(4))
    print('Gaussian coverage uses raw predicted sigma. Calibrated coverage uses separate calibration groups.')
    print('Missing calibrated coverage means no supported calibration interval. Nominal coverage is 0.90; transfer to unseen genes is empirical.')
    print('Bias, sigma, RMSE and NLL are in training-task-transformed units. Raw-score predictions and intervals are also saved.')
print(f'Evaluated {len(folds)} outer folds and {len(SEEDS)} seeds. Test results must not select checkpoints or tune this protocol.')
banner(f'DONE: {EXPERIMENT}')
if STORAGE == 'local':
    print('Download results before this runtime is deleted. Use the notebook download cell, or zip EXPERIMENT yourself.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. INSTALL UPDATED RELEASE
GPU: NVIDIA A100-SXM4-80GB
Release SHA256: 3ce6c2822684b45fde1a7cbe4505980c7b2fbf5736eeead0e69eb1f1bad9917a

2. FIX SPLITS AND CHECKPOINT SELECTION BEFORE TRAINING
Primary analysis excludes consensus and documented duplicate deposits; see v4/robust/build_report.json
Fold 0: test=['CYP2C9'], validation=['KRAS', 'MAGI2', 'MECP2', 'SRC', 'VIL1'], calibration=['HLA-A', 'KCNJ2', 'POU1F1'], train genes=481
Number of genes per task in each role:


,train,validation,calibration,test
abundance,13,1,2,1
activity,21,1,1,0
binding,6,1,1,1
fitness,28,1,0,0
stability,426,4,1,0


Fold 1: test=['GCK'], validation=['CYP2C9', 'SRC', 'TARDBP', 'TIAM1', 'TPK1'], calibration=['HLA-A', 'PTEN', 'TADA2A'], train genes=481
Number of genes per task in each role:


,train,validation,calibration,test
abundance,13,1,2,1
activity,21,1,1,0
binding,7,1,1,0
fitness,25,3,0,1
stability,427,3,1,0


Task coverage gap: fitness. Validation/calibration transfer for this task is not established.
Fold 2: test=['HLA-A'], validation=['CYP2C9', 'FLNB', 'SMURF2', 'SRC', 'ZNF81'], calibration=['KCNE1', 'KRAS', 'ZFP30'], train genes=481
Number of genes per task in each role:


...

,train,validation,calibration,test
abundance,15,1,1,0
activity,20,1,1,1
binding,7,1,1,0
fitness,27,1,0,1
stability,427,2,1,1


Task coverage gap: fitness. Validation/calibration transfer for this task is not established.
Fold 9: test=['VKORC1'], validation=['HLA-A', 'MDM2', 'NR3C2', 'PROP1', 'SRC'], calibration=['CYP2C9', 'FKRP', 'LSM14A'], train genes=481
Number of genes per task in each role:


,train,validation,calibration,test
abundance,14,1,1,1
activity,20,1,1,1
binding,7,1,1,0
fitness,28,1,0,0
stability,426,4,1,0


Outputs: /content/drive/MyDrive/MIPO_NDD/v4_validation_mechanism_650M_bafe6e1c78fd218e
120 requested model/seed/fold runs: ['mipo', 'esm_mlp', 'no_structure', 'no_global_kernel']
3000 batches x 8 = 24,000 sample draws/epoch; max 20 epochs; patience 6
Selection: mean assay Spearman within each validation gene, then equal mean across genes.
Expanded holdouts change training data; compare models within this new protocol.

3. REUSE OR EXTRACT FROZEN FEATURES
Local cache ready: /content/mipo_ndd_v4_validation_3ce6c2822684/v4_650M_masked

4. TRAIN AND RUN MATCHED ABLATIONS
Set aside partial run from torch 2.11.0+cu128 / Python 3.13.15; it restarts on torch 2.11.0+cu130 / Python 3.13.15. Kept at set_aside_partial_runs/mipo/seed_123/fold_02_20261002-092920
Set aside partial run from torch 2.11.0+cu128 / Python 3.13.15; it restarts on torch 2.11.0+cu130 / Python 3.13.15. Kept at set_aside_partial_runs/no_global_kernel/seed_123/fold_02_20261002-092925
Set aside partial run from torch 2.11.0+cu12

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. PROGRESS: safe to run at any time; reads Drive only.
import json, os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass
EXPERIMENT = Path(os.environ.get('MIPO_MYDRIVE', '/content/drive/MyDrive'))/'MIPO_NDD/v4_validation_mechanism_650M_bafe6e1c78fd218e'
protocol = json.loads((EXPERIMENT/'protocol.json').read_text())
total = len(protocol['models'])*len(protocol['seeds'])*len(protocol['folds'])
done = 0
for model in protocol['models']:
    for seed in protocol['seeds']:
        states = []
        for fold in protocol['folds']:
            run = EXPERIMENT/'runs'/model/f'seed_{seed}'/f'fold_{fold:02d}'
            if (run/'test_summary.json').exists():
                done += 1
                states.append('done')
            elif (run/'history.json').exists():
                states.append(f"{len(json.loads((run/'history.json').read_text()))}ep")
            else:
                states.append('-')
        print(f'{model:17s} seed {seed:<5d}', ' '.join(f'{s:>4s}' for s in states))
print(f'{done}/{total} runs finished')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mipo              seed 42    done done done done done done done done done done
mipo              seed 123      - done    - done done done done done done done
mipo              seed 2026  done done done done done done done done done done
esm_mlp           seed 42    done done done done done done done done done done
esm_mlp           seed 123   done done done done done done done done done done
esm_mlp           seed 2026  done done done done done done done done done done
no_structure      seed 42    done done done done    - done done done done done
no_structure      seed 123      - done    - done done done done done done done
no_structure      seed 2026  done done done done done done done done done done
no_global_kernel  seed 42    done done done done done done done done done done
no_global_kernel  seed 123      - done    - done done done done done done done
no

In [ ]:
import pandas as pd
R = '/content/drive/MyDrive/MIPO_NDD/v4_validation_mechanism_650M_bafe6e1c78fd218e/runs/'
for run in ['mipo/seed_123/fold_00', 'no_structure/seed_123/fold_00', 'no_global_kernel/seed_123/fold_00', 'no_structure/seed_42/fold_04']:
    d = pd.read_csv(R + run + '/validation_epochs/epoch_000.csv')
    bad = d[d.spearman.isna() | (d.prediction_std < 1e-4)]
    print(run, '->', len(bad), 'of', len(d), 'assays constant')
    print(bad[['gene', 'task', 'n', 'prediction_std']].to_string(index=False))


In [ ]:
# 5. DIAGNOSE the 7 failed runs, then train them with replacement seeds. Run AFTER cell 2 failed, in the same runtime.
# Why cell 2's retry cannot work: it reruns the SAME seed, and the sampler/init/dropout RNG are seeded from it
# (seed_all(seed+epoch)), so the run collapses identically. This cell finds out why, then trains new-seed replacements
# into runs_reseeded/ (never into runs/), so the DGX/Colab results are untouched and every substitution is on record.
RESEED_OFFSETS = (1000, 2000, 3000)    # replacement seed = original + offset; next one is tried only if the last collapsed
PARALLEL_RESEED = 5
LR_SCALE = 1.0                         # leave 1.0 (same protocol); <1 only if the diagnosis says saturation, and say so in the paper

import json, os, subprocess, sys, time
from pathlib import Path
import pandas as pd, psutil
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/MIPO_NDD')
EXPERIMENT = ROOT/'v4_validation_mechanism_650M_bafe6e1c78fd218e'
LOGS, RUNS, REPR = EXPERIMENT/'logs', EXPERIMENT/'runs', EXPERIMENT/'runs_reseeded'
projects = sorted(Path('/content').glob('mipo_ndd_v4_validation_*'), key=lambda p: p.stat().st_mtime)
assert projects and (projects[-1]/'active_v4_validation.json').exists(), \
    'Project folder is gone (runtime was reset). Run cells 1 and 2 first, then stop cell 2 once it prints "4. TRAIN".'
PROJECT = projects[-1]
CORPUS, RESOURCES = PROJECT/'v4/robust/curriculum.csv.gz', PROJECT/'v4/resources'
METADATA, SPLITS = PROJECT/'v4/robust/assay_metadata.csv', PROJECT/'active_mechanism_multigene_splits.json'
CACHE = PROJECT/'v4_650M_masked'
protocol = json.loads((EXPERIMENT/'protocol.json').read_text())
config = json.loads((EXPERIMENT/'config.json').read_text())
assert config['amp'] is False, 'config has amp=True: that is the earlier fp16 collapse'
try:
    beat = json.loads((EXPERIMENT/'.training_session.json').read_text())
    assert time.time()-beat['heartbeat'] > 600, f"{beat['session']} is training in this folder; stop it first"
except (OSError, ValueError):
    pass

def run_dir(model, seed, fold, base=RUNS):
    return base/model/f'seed_{seed}'/f'fold_{fold:02d}'

todo = [(m, s, f) for s in protocol['seeds'] for f in protocol['folds'] for m in protocol['models']
        if not (run_dir(m, s, f)/'test_summary.json').exists()]
print(f'{len(todo)} unfinished runs:', [f'{m}/s{s}/f{f}' for m, s, f in todo])

# ---- A. DIAGNOSE -------------------------------------------------------------------------------------------
print('\n=== A. WHY EACH RUN FAILED ===')
rows = []
for m, s, f in todo:
    d = run_dir(m, s, f)
    log = LOGS/f'{m}_seed{s}_fold{f:02d}.log'
    section = log.read_text(errors='replace').rsplit('\n=== ', 1)[-1] if log.exists() else ''
    lines = [l for l in section.splitlines() if l.strip() and not l.startswith('{"epoch"')]
    error = lines[-1][:140] if lines else 'no log'
    csvs = sorted((d/'validation_epochs').glob('epoch_*.csv'))
    if not csvs:
        rows.append(dict(run=f'{m}/s{s}/f{f}', error=error, epoch='-', assays='-', undefined='-', why='no validation csv'))
        continue
    v = pd.read_csv(csvs[-1])
    bad = v[v.spearman.isna()]
    why = {'n<3': int((bad.n < 3).sum()), 'constant target': int((bad.target_std == 0).sum()),
           'constant prediction': int((bad.prediction_std < 1e-4).sum())}
    rows.append(dict(run=f'{m}/s{s}/f{f}', error=error, epoch=csvs[-1].stem[-3:], assays=len(v), undefined=len(bad),
                     pred_std_median=round(float(v.prediction_std.median()), 5),
                     all_assays_flat=bool((v.prediction_std < 1e-4).all()), why=why))
    if len(bad):
        print(f'\n{m}/s{s}/f{f} epoch {csvs[-1].stem[-3:]}: undefined assays')
        display(bad[['gene', 'task', 'n', 'target_std', 'prediction_std', 'prediction_mean']].round(5))
diag = pd.DataFrame(rows)
display(diag)
diag.to_csv(EXPERIMENT/'failure_diagnosis.csv', index=False)
data_fault = [r for r in rows if isinstance(r.get('why'), dict) and (r['why']['n<3'] or r['why']['constant target'])]
if data_fault:
    print('\nSome validation assays are degenerate in the DATA (n<3 or constant target). A new seed cannot fix that; '
          'those folds need a validation-set fix, not a retrain. Not retraining:', [r['run'] for r in data_fault])
    todo = [t for t in todo if f'{t[0]}/s{t[1]}/f{t[2]}' not in {r['run'] for r in data_fault}]
else:
    print('\nVerdict: no data fault. The model emits exactly constant predictions on some validation assays, i.e. a training '
          'collapse that depends on the seed (the same folds finished with other seeds and esm_mlp). Replacing the seed.')

# ---- B. TRAIN REPLACEMENTS ---------------------------------------------------------------------------------
print('\n=== B. TRAIN REPLACEMENT SEEDS ===')
REPR.mkdir(exist_ok=True)
cfg_dir = PROJECT/'reseed_configs'
cfg_dir.mkdir(exist_ok=True)
cfg = dict(config)
cfg['lr'] = config['lr']*LR_SCALE
cfg_path = cfg_dir/f'config_lr{LR_SCALE}.json'
cfg_path.write_text(json.dumps(cfg, indent=2))
threads = str(max(1, (os.cpu_count() or 1)//PARALLEL_RESEED))
env = {**os.environ, 'OMP_NUM_THREADS': threads, 'MKL_NUM_THREADS': threads, 'PYTHONUNBUFFERED': '1'}
queue = [(job, 0) for job in todo]       # (job, attempt index)
running, outcome, last_launch = {}, {}, 0.
t0 = time.time()
while queue or running:
    for key, (proc, logf) in list(running.items()):
        code = proc.poll()
        if code is None:
            continue
        logf.close()
        del running[key]
        (m, s, f), k = key
        new_seed = s+RESEED_OFFSETS[k]
        log = LOGS/f'reseed_{m}_seed{new_seed}_fold{f:02d}.log'
        tail = log.read_text(errors='replace').rsplit('\n=== ', 1)[-1]
        if code == 0:
            outcome[(m, s, f)] = dict(status='finished', replacement_seed=new_seed, lr_scale=LR_SCALE)
            print(f'{time.strftime("%H:%M")} finished {m} seed {s}->{new_seed} fold {f}', flush=True)
        elif 'Validation Spearman undefined' in tail and k+1 < len(RESEED_OFFSETS):
            print(f'{time.strftime("%H:%M")} {m} seed {new_seed} fold {f} collapsed too; trying the next seed', flush=True)
            queue.insert(0, ((m, s, f), k+1))
        else:
            outcome[(m, s, f)] = dict(status='failed', last_seed=new_seed, log=str(log), tail=tail.strip().splitlines()[-1][:200])
            print(f'{time.strftime("%H:%M")} FAILED {m} seed {new_seed} fold {f}: {outcome[(m, s, f)]["tail"]}', flush=True)
    if queue and len(running) < PARALLEL_RESEED and time.time()-last_launch > 60 and \
            (not running or psutil.virtual_memory().available/1024**3 > 12):
        (m, s, f), k = queue.pop(0)
        new_seed = s+RESEED_OFFSETS[k]
        logf = open(LOGS/f'reseed_{m}_seed{new_seed}_fold{f:02d}.log', 'a')
        logf.write(f"\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} reseed of {m} seed {s} fold {f}\n")
        logf.flush()
        cmd = [sys.executable, PROJECT/'scripts'/'run_experiments.py', '--corpus', CORPUS, '--resources', RESOURCES,
               '--cache', CACHE, '--splits', SPLITS, '--config', cfg_path, '--metadata', METADATA, '--out', REPR,
               '--folds', f, '--seeds', new_seed, '--ablations', m]
        running[((m, s, f), k)] = (subprocess.Popen(list(map(str, cmd)), stdout=logf, stderr=subprocess.STDOUT, env=env,
                                                    cwd=PROJECT), logf)
        last_launch = time.time()
        print(f'{time.strftime("%H:%M")} started {m} seed {s}->{new_seed} fold {f} ({len(running)} running)', flush=True)
    time.sleep(20)
print(f'Run time: {(time.time()-t0)/60:.1f} min')
(EXPERIMENT/'reseeded_runs.json').write_text(json.dumps(
    {f'{m}/seed_{s}/fold_{f:02d}': v for (m, s, f), v in outcome.items()}, indent=2))
display(pd.DataFrame([dict(run=f'{m}/s{s}/f{f}', **v) for (m, s, f), v in outcome.items()]))
print('Replacements are in runs_reseeded/ and mapped in reseeded_runs.json. They are NOT in runs/, so cells 6-7 do not '
      'include them. Report any substitution explicitly: dropping or re-rolling collapsed seeds biases a model upward. '
      'Also report how many original-seed runs collapsed, per model.')


7 unfinished runs: ['no_structure/s42/f4', 'mipo/s123/f0', 'no_structure/s123/f0', 'no_global_kernel/s123/f0', 'mipo/s123/f2', 'no_structure/s123/f2', 'no_global_kernel/s123/f2']

=== A. WHY EACH RUN FAILED ===

no_structure/s42/f4 epoch 000: undefined assays


,gene,task,n,target_std,prediction_std,prediction_mean
3,KRAS,abundance,3066,0.64492,0.0,-0.12390
5,SRC,activity,3372,0.99184,0.0,-0.20874
6,SRC,activity,3637,0.69081,0.0,-0.20725
7,SRC,fitness,3366,0.95458,0.0,-0.01988



mipo/s123/f0 epoch 000: undefined assays


,gene,task,n,target_std,prediction_std,prediction_mean
0,KRAS,abundance,3066,0.64492,0.0,-0.18017
1,KRAS,binding,3084,0.27510,0.0,0.12365
2,MAGI2,stability,1643,1.15885,0.0,-0.15437
3,MAGI2,stability,1583,0.89902,0.0,-0.15437
4,MAGI2,stability,1436,0.80030,0.0,-0.15437
5,MAGI2,stability,597,0.96732,0.0,-0.15437
6,MECP2,stability,1364,1.02721,0.0,-0.15437
7,SRC,activity,3372,1.05410,0.0,-0.13080
8,SRC,activity,3637,0.73418,0.0,-0.14687
9,SRC,fitness,3366,0.95458,0.0,0.00534



no_structure/s123/f0 epoch 000: undefined assays


...

,gene,task,n,target_std,prediction_std,prediction_mean
0,KRAS,abundance,3066,0.64492,0.0,-0.18017
1,KRAS,binding,3084,0.27510,0.0,0.12365
2,MAGI2,stability,1643,1.15885,0.0,-0.15437
3,MAGI2,stability,1583,0.89902,0.0,-0.15437
4,MAGI2,stability,1436,0.80030,0.0,-0.15437
5,MAGI2,stability,597,0.96732,0.0,-0.15437
6,MECP2,stability,1364,1.02721,0.0,-0.15437
7,SRC,activity,3372,1.05410,0.0,-0.13080
8,SRC,activity,3637,0.73418,0.0,-0.14687
9,SRC,fitness,3366,0.95458,0.0,0.00534


,run,error,epoch,assays,undefined,pred_std_median,all_assays_flat,why
0,no_structure/s42/f4,Sampling: task-balanced over 5 tasks; 0 two-ta...,000,10,4,0.0,True,"{'n<3': 0, 'constant target': 0, 'constant pre..."
1,mipo/s123/f0,Sampling: task-balanced over 5 tasks; 0 two-ta...,000,12,12,0.0,True,"{'n<3': 0, 'constant target': 0, 'constant pre..."
2,no_structure/s123/f0,ValueError: Validation Spearman undefined for ...,000,12,12,0.0,True,"{'n<3': 0, 'constant target': 0, 'constant pre..."
3,no_global_kernel/s123/f0,ValueError: Validation Spearman undefined for ...,000,12,12,0.0,True,"{'n<3': 0, 'constant target': 0, 'constant pre..."
4,mipo/s123/f2,ValueError: Validation Spearman undefined for ...,-,-,-,NaN,NaN,no validation csv
5,no_structure/s123/f2,ValueError: Validation Spearman undefined for ...,-,-,-,NaN,NaN,no validation csv
6,no_global_kernel/s123/f2,ValueError: Validation Spearman undefined for ...,-,-,-,NaN,NaN,no validation csv



Verdict: no data fault. The model emits exactly constant predictions on some validation assays, i.e. a training collapse that depends on the seed (the same folds finished with other seeds and esm_mlp). Replacing the seed.

=== B. TRAIN REPLACEMENT SEEDS ===
09:33 started no_structure seed 42->1042 fold 4 (1 running)
09:33 finished no_structure seed 42->1042 fold 4
09:34 started mipo seed 123->1123 fold 0 (1 running)
09:34 finished mipo seed 123->1123 fold 0
09:35 started no_structure seed 123->1123 fold 0 (1 running)
09:35 finished no_structure seed 123->1123 fold 0
09:36 started no_global_kernel seed 123->1123 fold 0 (1 running)
09:36 finished no_global_kernel seed 123->1123 fold 0
09:37 started mipo seed 123->1123 fold 2 (1 running)
09:38 started no_structure seed 123->1123 fold 2 (2 running)
09:39 started no_global_kernel seed 123->1123 fold 2 (3 running)
09:47 mipo seed 1123 fold 2 collapsed too; trying the next seed
09:47 started mipo seed 123->2123 fold 2 (3 running)
09:47 no_st

,run,status,replacement_seed,lr_scale,last_seed,log,tail
0,no_structure/s42/f4,finished,1042.0,1.0,NaN,NaN,NaN
1,mipo/s123/f0,finished,1123.0,1.0,NaN,NaN,NaN
2,no_structure/s123/f0,finished,1123.0,1.0,NaN,NaN,NaN
3,no_global_kernel/s123/f0,finished,1123.0,1.0,NaN,NaN,NaN
4,mipo/s123/f2,finished,2123.0,1.0,NaN,NaN,NaN
5,no_structure/s123/f2,finished,2123.0,1.0,NaN,NaN,NaN
6,no_global_kernel/s123/f2,failed,NaN,NaN,2123.0,/content/drive/MyDrive/MIPO_NDD/v4_validation_...,ValueError: Resume provenance mismatch; restor...


Replacements are in runs_reseeded/ and mapped in reseeded_runs.json. They are NOT in runs/, so cells 6-7 do not include them. Report any substitution explicitly: dropping or re-rolling collapsed seeds biases a model upward. Also report how many original-seed runs collapsed, per model.


In [ ]:
# 6. Finish the last run: no_global_kernel, original seed 123, fold 2. Run in the same runtime, after cell 5.
import json, platform, shutil, subprocess, sys, time
import torch
assert 'cfg_path' in globals(), 'Run cell 5 first (it defines the paths this cell uses)'
MODEL, ORIG_SEED, FOLD = 'no_global_kernel', 123, 2
SEEDS_TO_TRY = (2123, 3123)

# ---- why the partial run cannot resume --------------------------------------------------------------------
partial = run_dir(MODEL, SEEDS_TO_TRY[0], FOLD, REPR)
if (partial/'last.pt').exists():
    saved = torch.load(partial/'last.pt', map_location='cpu', weights_only=True)
    prov = saved['provenance']
    print(f"Partial run: {saved['epoch']+1} epochs done, best validation spearman {saved['best']:.3f}")
    print('saved torch/python:', prov['torch'], prov['python'], '| now:', torch.__version__, platform.python_version())
    from mipo.train import code_fingerprints
    print('code files changed:', [k for k, v in code_fingerprints().items() if prov['code_sha256'].get(k) != v] or 'none')
    current = json.loads(cfg_path.read_text())
    skip = {'seed', 'model', 'structure', 'global_kernel'}
    print('config keys that differ:', [k for k in current if k not in skip and current[k] != prov['config'].get(k)] or 'none')
    target = (EXPERIMENT/'set_aside_partial_runs'/'reseeded'/
              f"{MODEL}_seed{SEEDS_TO_TRY[0]}_fold{FOLD:02d}_{time.strftime('%Y%m%d-%H%M%S')}")
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(partial), str(target))
    print('Set aside (kept):', target.relative_to(EXPERIMENT))

# ---- retrain from epoch 0 ---------------------------------------------------------------------------------
result = None
for seed in SEEDS_TO_TRY:
    log = LOGS/f'reseed_{MODEL}_seed{seed}_fold{FOLD:02d}.log'
    cmd = [sys.executable, PROJECT/'scripts'/'run_experiments.py', '--corpus', CORPUS, '--resources', RESOURCES,
           '--cache', CACHE, '--splits', SPLITS, '--config', cfg_path, '--metadata', METADATA, '--out', REPR,
           '--folds', FOLD, '--seeds', seed, '--ablations', MODEL]
    shown = 0
    with open(log, 'a') as lf:
        lf.write(f"\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} reseed of {MODEL} seed {ORIG_SEED} fold {FOLD}, seed {seed}\n")
        lf.flush()
        proc = subprocess.Popen(list(map(str, cmd)), stdout=lf, stderr=subprocess.STDOUT, env=env, cwd=PROJECT)
        t0 = time.time()
        while proc.poll() is None:
            time.sleep(60)
            lines = [l for l in log.read_text(errors='replace').rsplit('\n=== ', 1)[-1].splitlines() if l.startswith('{"epoch"')]
            for l in lines[shown:]:
                try:
                    r = json.loads(l)
                    print(f"{time.strftime('%H:%M')} seed {seed} epoch {r['epoch']}: val spearman {r['validation_spearman']:.3f} "
                          f"(bad_epochs {r['bad_epochs']}/6)", flush=True)
                except ValueError:
                    pass
            shown = len(lines)
    tail = log.read_text(errors='replace').rsplit('\n=== ', 1)[-1]
    if proc.returncode == 0:
        result = dict(status='finished', replacement_seed=seed, lr_scale=LR_SCALE,
                      note='restarted from epoch 0 after a partial run on another torch/Python')
        print(f'finished seed {seed} in {(time.time()-t0)/60:.0f} min')
        break
    if 'Validation Spearman undefined' in tail:
        print(f'seed {seed} collapsed; trying the next seed', flush=True)
        continue
    raise RuntimeError(f'seed {seed} failed: {tail.strip().splitlines()[-1][:300]}')

path = EXPERIMENT/'reseeded_runs.json'
done = json.loads(path.read_text()) if path.exists() else {}
done[f'{MODEL}/seed_{ORIG_SEED}/fold_{FOLD:02d}'] = result or dict(status='failed', tried=list(SEEDS_TO_TRY))
path.write_text(json.dumps(done, indent=2))
print(json.dumps(done, indent=2))


Partial run: 6 epochs done, best validation spearman 0.537
saved torch/python: 2.11.0+cu130 3.13.15 | now: 2.11.0+cu130 3.13.15
code files changed: none
config keys that differ: none
Set aside (kept): set_aside_partial_runs/reseeded/no_global_kernel_seed2123_fold02_20261002-110216
11:12 seed 2123 epoch 0: val spearman 0.486 (bad_epochs 0/6)


KeyboardInterrupt: 

In [ ]:
# 7. Finish the last run (no_global_kernel, seed 123, fold 2). Resumes a usable partial run instead of discarding it.
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path
import psutil, torch

MODEL, ORIG_SEED, FOLD, SEEDS_TO_TRY, LR_SCALE = 'no_global_kernel', 123, 2, (2123, 3123), 1.0
ROOT = Path('/content/drive/MyDrive/MIPO_NDD')
EXPERIMENT = ROOT/'v4_validation_mechanism_650M_bafe6e1c78fd218e'
LOGS, REPR, ASIDE = EXPERIMENT/'logs', EXPERIMENT/'runs_reseeded', EXPERIMENT/'set_aside_partial_runs'
projects = sorted(Path('/content').glob('mipo_ndd_v4_validation_*'), key=lambda p: p.stat().st_mtime)
assert projects and (projects[-1]/'active_v4_validation.json').exists(), \
    'Runtime was reset: run cells 1 and 2, then stop cell 2 once it prints "4. TRAIN"'
PROJECT = projects[-1]
CORPUS, RESOURCES = PROJECT/'v4/robust/curriculum.csv.gz', PROJECT/'v4/resources'
METADATA, SPLITS, CACHE = PROJECT/'v4/robust/assay_metadata.csv', PROJECT/'active_mechanism_multigene_splits.json', PROJECT/'v4_650M_masked'
cfg_path = PROJECT/'reseed_configs'/'config_lr1.0.json'
if not cfg_path.exists():
    cfg_path.parent.mkdir(exist_ok=True)
    cfg_path.write_text((EXPERIMENT/'config.json').read_text())
env = {**os.environ, 'OMP_NUM_THREADS': str(os.cpu_count() or 1), 'MKL_NUM_THREADS': str(os.cpu_count() or 1), 'PYTHONUNBUFFERED': '1'}

# 1. never start a second trainer on top of an orphan
live = [p.info['pid'] for p in psutil.process_iter(['pid', 'cmdline']) if 'run_experiments.py' in ' '.join(p.info['cmdline'] or [])]
if live:
    raise RuntimeError(f'run_experiments is already running (pid {live}), probably left by an interrupted cell. '
                       'Check its log, or stop it in the Terminal with: kill ' + ' '.join(map(str, live)) + ' , then run this cell again.')

# 2. reuse the furthest partial run that this torch/Python can resume
now = [str(torch.__version__), platform.python_version()]
def info(d):
    try:
        s = torch.load(d/'last.pt', map_location='cpu', weights_only=True)
        return s['epoch']+1, [s['provenance']['torch'], s['provenance']['python']]
    except Exception:
        return 0, None
out = REPR/MODEL/f'seed_{SEEDS_TO_TRY[0]}'/f'fold_{FOLD:02d}'
if (out/'last.pt').exists():
    epochs_done, tp = info(out)
    if tp == now:
        print(f'Resuming the partial run in place ({epochs_done} epochs done).')
    else:
        target = ASIDE/'reseeded'/f"{MODEL}_seed{SEEDS_TO_TRY[0]}_fold{FOLD:02d}_{time.strftime('%Y%m%d-%H%M%S')}"
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(out), str(target))
        print(f'Partial run is from torch/Python {tp}, now {now}: set aside, restarting from epoch 0.')
else:
    cands = []
    for last in ASIDE.rglob('last.pt'):
        if last.parent.name.startswith(f'{MODEL}_seed{SEEDS_TO_TRY[0]}_fold{FOLD:02d}'):
            e, tp = info(last.parent)
            if tp == now:
                cands.append((e, last.parent))
    if cands:
        e, src = max(cands, key=lambda x: x[0])
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.exists():                # leftover stub from an interrupted attempt (no last.pt); keep it, don't merge
            stub = ASIDE/'reseeded'/f"{MODEL}_seed{SEEDS_TO_TRY[0]}_fold{FOLD:02d}_stub_{time.strftime('%Y%m%d-%H%M%S')}"
            shutil.move(str(out), str(stub))
        shutil.copytree(src, out)       # copied, so the set-aside record stays

        print(f'Restored a {e}-epoch partial run from {src.relative_to(ASIDE)}; it resumes on this runtime.')
    else:
        print('No partial run usable on this torch/Python; starting from epoch 0.')

# 3. train, printing each epoch; an interrupt stops the child so nothing is left running
result = None
for seed in SEEDS_TO_TRY:
    log = LOGS/f'reseed_{MODEL}_seed{seed}_fold{FOLD:02d}.log'
    cmd = [sys.executable, PROJECT/'scripts'/'run_experiments.py', '--corpus', CORPUS, '--resources', RESOURCES,
           '--cache', CACHE, '--splits', SPLITS, '--config', cfg_path, '--metadata', METADATA, '--out', REPR,
           '--folds', FOLD, '--seeds', seed, '--ablations', MODEL]
    shown, t0 = 0, time.time()
    with open(log, 'a') as lf:
        lf.write(f"\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} reseed of {MODEL} seed {ORIG_SEED} fold {FOLD}, seed {seed}\n")
        lf.flush()
        proc = subprocess.Popen(list(map(str, cmd)), stdout=lf, stderr=subprocess.STDOUT, env=env, cwd=PROJECT)
        try:
            while proc.poll() is None:
                time.sleep(60)
                lines = [l for l in log.read_text(errors='replace').rsplit('\n=== ', 1)[-1].splitlines() if l.startswith('{"epoch"')]
                for l in lines[shown:]:
                    try:
                        r = json.loads(l)
                        print(f"{time.strftime('%H:%M')} seed {seed} epoch {r['epoch']}: val spearman "
                              f"{r['validation_spearman']:.3f} (bad_epochs {r['bad_epochs']}/6)", flush=True)
                    except ValueError:
                        pass
                shown = len(lines)
        finally:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(60)
                except subprocess.TimeoutExpired:
                    proc.kill()
                print('Interrupted: training stopped; run this cell again to resume from the last finished epoch.')
    tail = log.read_text(errors='replace').rsplit('\n=== ', 1)[-1]
    if proc.returncode == 0:
        result = dict(status='finished', replacement_seed=seed, lr_scale=LR_SCALE,
                      note='resumed or restarted after interruptions/partial runs from another torch/Python')
        print(f'finished seed {seed}')
        break
    if 'Validation Spearman undefined' in tail:
        print(f'seed {seed} collapsed; trying the next seed', flush=True)
        continue
    raise RuntimeError(f'seed {seed} stopped: {tail.strip().splitlines()[-1][:300]}')

path = EXPERIMENT/'reseeded_runs.json'
done = json.loads(path.read_text()) if path.exists() else {}
done[f'{MODEL}/seed_{ORIG_SEED}/fold_{FOLD:02d}'] = result or dict(status='failed', tried=list(SEEDS_TO_TRY))
path.write_text(json.dumps(done, indent=2))
print(json.dumps(done[f'{MODEL}/seed_{ORIG_SEED}/fold_{FOLD:02d}'], indent=2))


Restored a 6-epoch partial run from reseeded/no_global_kernel_seed2123_fold02_20261002-110216; it resumes on this runtime.
11:52 seed 2123 epoch 6: val spearman 0.554 (bad_epochs 0/6)
12:00 seed 2123 epoch 7: val spearman 0.553 (bad_epochs 1/6)
12:08 seed 2123 epoch 8: val spearman 0.545 (bad_epochs 2/6)
12:17 seed 2123 epoch 9: val spearman 0.559 (bad_epochs 0/6)
12:25 seed 2123 epoch 10: val spearman 0.560 (bad_epochs 0/6)
12:33 seed 2123 epoch 11: val spearman 0.567 (bad_epochs 0/6)
12:42 seed 2123 epoch 12: val spearman 0.570 (bad_epochs 0/6)
12:50 seed 2123 epoch 13: val spearman 0.567 (bad_epochs 1/6)
12:59 seed 2123 epoch 14: val spearman 0.569 (bad_epochs 2/6)
13:07 seed 2123 epoch 15: val spearman 0.566 (bad_epochs 3/6)
13:16 seed 2123 epoch 16: val spearman 0.568 (bad_epochs 4/6)
13:24 seed 2123 epoch 17: val spearman 0.569 (bad_epochs 5/6)
13:32 seed 2123 epoch 18: val spearman 0.569 (bad_epochs 6/6)
finished seed 2123
{
  "status": "finished",
  "replacement_seed": 2123,
  

In [ ]:
# Make a light results zip (metrics, predictions, histories, logs, protocol; no .pt checkpoints) on Drive.
import shutil, zipfile
from pathlib import Path
EXP = Path('/content/drive/MyDrive/MIPO_NDD/v4_validation_mechanism_650M_bafe6e1c78fd218e')
OUT = EXP.parent/'MIPO_NDD_results_light.zip'
INCLUDE_CHECKPOINTS = False          # True also adds best.pt/last.pt (large)
n = 0
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EXP.rglob('*')):
        if not p.is_file() or p.name.startswith('.'):
            continue
        if p.suffix == '.pt' and not INCLUDE_CHECKPOINTS:
            continue
        z.write(p, p.relative_to(EXP.parent))
        n += 1
print(f'{n} files, {OUT.stat().st_size/1e6:.0f} MB -> {OUT}')


4765 files, 239 MB -> /content/drive/MyDrive/MIPO_NDD/MIPO_NDD_results_light.zip


In [ ]:
# 8. FINAL ANALYSIS from runs/ + runs_reseeded/. Replacement runs are labelled; summaries with and without them.
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/MIPO_NDD')
EXPERIMENT = ROOT/'v4_validation_mechanism_650M_bafe6e1c78fd218e'
RUNS, REPR = EXPERIMENT/'runs', EXPERIMENT/'runs_reseeded'
projects = sorted(Path('/content').glob('mipo_ndd_v4_validation_*'), key=lambda p: p.stat().st_mtime)
assert projects and (projects[-1]/'active_v4_validation.json').exists(), \
    'Runtime was reset: run cells 1 and 2, then stop cell 2 once it prints "4. TRAIN"'
PROJECT = projects[-1]
CORPUS, RESOURCES, CACHE = PROJECT/'v4/robust/curriculum.csv.gz', PROJECT/'v4/resources', PROJECT/'v4_650M_masked'
panel = pd.read_csv(PROJECT/'v4/robust/mechanism_panel.csv').set_index('gene')

def sh(*args):
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, cwd=PROJECT)
    return r

# 1. exact-row cached LLR baseline, on both run folders (writes matched_llr_metrics.csv next to each run)
for base in (RUNS, REPR):
    r = sh(PROJECT/'scripts'/'compare_cached_llr.py', '--corpus', CORPUS, '--resources', RESOURCES, '--cache', CACHE, '--runs', base)
    print(base.name, 'LLR baseline:', 'ok' if r.returncode == 0 else 'FAILED\n'+r.stderr[-1500:])
    if r.returncode:
        raise RuntimeError('Fix this before interpreting results')

# 2. merge; a replacement run keeps the ORIGINAL seed label (1042->42, 1123/2123->123) plus a flag
rows = []
for base, is_re in ((RUNS, False), (REPR, True)):
    for f in sorted(base.glob('*/seed_*/fold_*/matched_llr_metrics.csv')):
        seed = int(f.parents[1].name.split('_')[1])
        d = pd.read_csv(f)
        d.insert(0, 'model', f.parents[2].name)
        d.insert(1, 'seed', seed % 1000 if is_re else seed)
        d.insert(2, 'fold', int(f.parent.name.split('_')[1]))
        d['replacement'], d['replacement_seed'] = is_re, (seed if is_re else None)
        rows.append(d)
table = pd.concat(rows, ignore_index=True)
slots = table.groupby(['model', 'seed', 'fold']).replacement.nunique()
assert (slots == 1).all(), f'Some model/seed/fold slots appear in both runs/ and runs_reseeded/: {slots[slots > 1].index.tolist()}'
table['evaluation_part'] = table.gene.map(panel.part)
table['previously_examined'] = table.gene.map(panel.previously_examined)
table.to_csv(EXPERIMENT/'matched_comparison_all.csv', index=False)
n_slots = table.groupby(['model', 'seed', 'fold']).ngroups
print(f'{n_slots} model/seed/fold slots; {table[table.replacement].groupby(["model","seed","fold"]).ngroups} are replacement-seed runs')
if table[['model_spearman', 'llr_spearman']].isna().any().any():
    print('WARNING: some correlations are undefined; inspect matched_comparison_all.csv. Summaries below skip nothing silently.')

# 3. how often each model collapsed (original runs that needed a new seed)
slot_info = table.drop_duplicates(['model', 'seed', 'fold'])
collapse = slot_info.groupby('model').agg(slots=('fold', 'size'), collapsed_and_replaced=('replacement', 'sum'))
collapse['collapse_rate'] = (collapse.collapsed_and_replaced/collapse.slots).round(3)
print('\nCollapse rate of ORIGINAL runs per model (report this):')
display(collapse)
collapse.to_csv(EXPERIMENT/'collapse_rate_by_model.csv')

# 4. macro summaries: assays within gene, then equal gene weight, then mean/std across seeds
def macro(t):
    gene = t.groupby(['model', 'seed', 'evaluation_part', 'gene'])[['model_spearman', 'llr_spearman', 'delta']].mean()
    per_seed = gene.groupby(['model', 'seed', 'evaluation_part']).mean()
    return per_seed.groupby(['model', 'evaluation_part']).agg(['mean', 'std']).round(4)
with_repl, originals = macro(table), macro(table[~table.replacement])
print('\nA. ALL slots (replacement-seed runs included):')
display(with_repl)
print('B. ORIGINAL runs only (collapsed slots dropped; sensitivity check, unbalanced across models):')
display(originals)
with_repl.to_csv(EXPERIMENT/'macro_comparison_with_replacements.csv')
originals.to_csv(EXPERIMENT/'macro_comparison_originals_only.csv')

gene_tab = table.groupby(['evaluation_part', 'gene', 'model']).model_spearman.mean().unstack('model')
gene_tab['LLR'] = table.groupby(['evaluation_part', 'gene']).llr_spearman.mean()
print('\nPer-gene mean Spearman (assays averaged within gene, then across seeds):')
display(gene_tab.round(3))
gene_tab.to_csv(EXPERIMENT/'per_gene_comparison.csv')

# 5. paired property contrasts (mechanism-separation diagnostic), one file per run folder
for base, name in ((RUNS, 'paired_property_contrasts.csv'), (REPR, 'paired_property_contrasts_reseeded.csv')):
    r = sh(PROJECT/'scripts'/'evaluate_property_contrasts.py', '--runs', base, '--out', EXPERIMENT/name)
    print(name, 'ok' if r.returncode == 0 else 'FAILED: '+r.stderr[-400:])
print('\nDone. Test results must not be used to pick checkpoints, seeds or settings. Report the collapse rates and the '
      '7 substituted runs alongside table A.')


runs LLR baseline: ok
runs_reseeded LLR baseline: ok
120 model/seed/fold slots; 7 are replacement-seed runs

Collapse rate of ORIGINAL runs per model (report this):


,slots,collapsed_and_replaced,collapse_rate
model,,,
esm_mlp,30,0,0.000
mipo,30,2,0.067
no_global_kernel,30,2,0.067
no_structure,30,3,0.100



A. ALL slots (replacement-seed runs included):


model_spearman         llr_spearman       \
                                           mean     std         mean  std   
model            evaluation_part                                            
esm_mlp          mechanism_only          0.4427  0.0136       0.4719  0.0   
                 ndd_evaluation          0.4696  0.0033       0.4256  0.0   
mipo             mechanism_only          0.4636  0.0038       0.4719  0.0   
                 ndd_evaluation          0.4671  0.0035       0.4256  0.0   
no_global_kernel mechanism_only          0.4610  0.0083       0.4719  0.0   
                 ndd_evaluation          0.4690  0.0072       0.4256  0.0   
no_structure     mechanism_only          0.4593  0.0140       0.4719  0.0   
                 ndd_evaluation          0.4670  0.0086       0.4256  0.0   

                                   delta          
                                    mean     std  
model            evaluation_part                  
esm_mlp          mechanism_only  -0.0292  0.0136  
                 ndd_evaluation   0.0440  0.0033  
mipo             mechanism_only  -0.0083  0.0038  
                 ndd_evaluation   0.0415  0.0035  
no_global_kernel mechanism_only  -0.0109  0.0083  
                 ndd_evaluation   0.0434  0.0072  
no_structure     mechanism_only  -0.0126  0.0140  
                 ndd_evaluation   0.0414  0.0086

B. ORIGINAL runs only (collapsed slots dropped; sensitivity check, unbalanced across models):


model_spearman         llr_spearman          \
                                           mean     std         mean     std   
model            evaluation_part                                               
esm_mlp          mechanism_only          0.4427  0.0136       0.4719  0.0000   
                 ndd_evaluation          0.4696  0.0033       0.4256  0.0000   
mipo             mechanism_only          0.4578  0.0077       0.4633  0.0149   
                 ndd_evaluation          0.4671  0.0035       0.4256  0.0000   
no_global_kernel mechanism_only          0.4554  0.0132       0.4633  0.0149   
                 ndd_evaluation          0.4690  0.0072       0.4256  0.0000   
no_structure     mechanism_only          0.4562  0.0270       0.4683  0.0206   
                 ndd_evaluation          0.4670  0.0086       0.4256  0.0000   

                                   delta          
                                    mean     std  
model            evaluation_part                  
esm_mlp          mechanism_only  -0.0292  0.0136  
                 ndd_evaluation   0.0440  0.0033  
mipo             mechanism_only  -0.0055  0.0081  
                 ndd_evaluation   0.0415  0.0035  
no_global_kernel mechanism_only  -0.0080  0.0095  
                 ndd_evaluation   0.0434  0.0072  
no_structure     mechanism_only  -0.0121  0.0144  
                 ndd_evaluation   0.0414  0.0086


Per-gene mean Spearman (assays averaged within gene, then across seeds):


model                    esm_mlp   mipo  no_global_kernel  no_structure    LLR
evaluation_part gene                                                          
mechanism_only  CYP2C9     0.679  0.682             0.649         0.649  0.657
                GCK        0.457  0.457             0.451         0.453  0.442
                HLA-A      0.327  0.357             0.381         0.393  0.442
                KCNE1      0.169  0.277             0.253         0.226  0.315
                KCNJ2      0.352  0.352             0.355         0.358  0.367
                SLC22A1    0.603  0.599             0.612         0.593  0.601
                SRC        0.503  0.526             0.520         0.526  0.490
                VKORC1     0.451  0.459             0.466         0.476  0.462
ndd_evaluation  KRAS       0.414  0.408             0.404         0.402  0.359
                PTEN       0.525  0.526             0.534         0.532  0.492

paired_property_contrasts.csv ok
paired_property_contrasts_reseeded.csv ok

Done. Test results must not be used to pick checkpoints, seeds or settings. Report the collapse rates and the 7 substituted runs alongside table A.


In [ ]:
# 9. DIAGNOSIS (existing outputs + seed-42 checkpoints; no training). Three questions:
#   A. mechanism separation: does a model predict which variants hurt one assay more than the other?
#   B. does the fusion actually use the structure pieces, or ignore them?
#   C. does any architecture help on one task type that the averages hide?
from pathlib import Path
import numpy as np, pandas as pd, torch
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/MIPO_NDD')
EXPERIMENT = ROOT/'v4_validation_mechanism_650M_bafe6e1c78fd218e'
RUNS, REPR = EXPERIMENT/'runs', EXPERIMENT/'runs_reseeded'
projects = sorted(Path('/content').glob('mipo_ndd_v4_validation_*'), key=lambda p: p.stat().st_mtime)
assert projects and (projects[-1]/'active_v4_validation.json').exists(), \
    'Runtime was reset: run cells 1 and 2, then stop cell 2 once it prints "4. TRAIN"'
PROJECT = projects[-1]
CORPUS, RESOURCES, CACHE = PROJECT/'v4/robust/curriculum.csv.gz', PROJECT/'v4/resources', PROJECT/'v4_650M_masked'
METADATA = PROJECT/'v4/robust/assay_metadata.csv'

# ---- A. mechanism separation ------------------------------------------------------------------------------
print('=== A. MECHANISM SEPARATION (two assays on the same variants, e.g. abundance vs activity) ===')
pc = pd.concat([pd.read_csv(EXPERIMENT/'paired_property_contrasts.csv').assign(replacement=False),
                pd.read_csv(EXPERIMENT/'paired_property_contrasts_reseeded.csv').assign(replacement=True)], ignore_index=True)
pc.loc[pc.replacement, 'seed'] = pc.loc[pc.replacement, 'seed'] % 1000
print('pair status counts:', pc.status.value_counts().to_dict())
# rank_contrast_spearman: does the predicted (assay A - assay B) rank difference track the observed one? 0 = no separation.
# mae_gain > 0: beats predicting "no difference", which is all a single-score predictor such as the LLR can do.
pc['mae_gain'] = 1 - pc.rank_contrast_mae/pc.zero_contrast_mae
by_gene = pc.groupby(['model', 'seed', 'part', 'gene'])[['rank_contrast_spearman', 'mae_gain']].mean()
per_seed = by_gene.groupby(level=['model', 'seed', 'part']).mean()
display(per_seed.groupby(level=['model', 'part']).agg(['mean', 'std']).round(3))
print('Abundance-anchored pairs only (stability proxy vs function):')
anch = (pc[pc.abundance_anchored].groupby(['model', 'seed', 'gene']).rank_contrast_spearman.mean()
        .groupby(level=['model', 'seed']).mean().groupby(level='model').agg(['mean', 'std']).round(3))
display(anch)
gene_view = by_gene.rank_contrast_spearman.groupby(level=['part', 'gene', 'model']).mean().unstack('model').round(3)
print('Per gene (mean over pairs and seeds):')
display(gene_view)
gene_view.to_csv(EXPERIMENT/'diagnosis_contrasts_by_gene.csv')

# ---- B. what the fusion uses --------------------------------------------------------------------------------
print('\n=== B. WHAT THE FUSION USES (seed-42 best checkpoints, up to 2000 test variants per gene) ===')
from mipo.train import restore, loader
from mipo.common import to_device
from mipo.data import FeatureStore, TargetTransform, Metadata
from mipo.corpus import read_corpus
device = 'cuda' if torch.cuda.is_available() else 'cpu'
corpus, store = read_corpus(CORPUS), FeatureStore(RESOURCES, CACHE)
PIECES = ['local (seq window + field at site)', 'field pool (whole-protein field)', 'global context', 'mutation impulse + LLR']
records = []
for model_name in ['esm_mlp', 'mipo', 'no_structure', 'no_global_kernel']:
    ckpts = {}
    for base, seed in ((REPR, 1042), (RUNS, 42)):          # original seed-42 run wins; replacement only if it collapsed
        for p in base.glob(f'{model_name}/seed_{seed}/fold_*/best.pt'):
            if (p.parent/'test_summary.json').exists():
                ckpts[int(p.parent.name.split('_')[1])] = p
    for fold, ckpt in sorted(ckpts.items()):
        model, state = restore(ckpt, device)
        model.eval()
        prov = state['provenance']
        config = dict(prov['config'], batch_size=64)
        rows = corpus[corpus.gene.isin(prov['fold']['test'])]
        rows = rows.groupby('gene', group_keys=False).apply(lambda g: g.sample(min(len(g), 2000), random_state=0))
        dl = loader(rows, store, TargetTransform(state['transform']), Metadata(METADATA, state['metadata_vocab']), config)
        gene_of = dict(zip(dl.dataset.rows.row_id, dl.dataset.rows.gene))
        captured, g_all, s_all, genes = [], [], [], []
        hook = model.fusion.register_forward_pre_hook(lambda m, inp: captured.append(inp[0].detach()))
        with torch.inference_mode():
            for b in dl:
                ids = torch.as_tensor(b['row_id']).cpu().numpy()
                out = model(to_device(b, device))
                x = captured.pop().float().reshape(len(ids), 4, -1).norm(dim=-1)
                s_all.append((x/x.sum(-1, keepdim=True)).cpu().numpy())
                g_all.append(out['gates'].float().cpu().numpy())
                genes += [gene_of[i] for i in ids]
        hook.remove()
        df = pd.DataFrame(np.hstack([np.vstack(g_all), np.vstack(s_all)]),
                          columns=[f'gate | {p}' for p in PIECES]+[f'share | {p}' for p in PIECES])
        df['gene'] = genes
        m = df.groupby('gene').mean().reset_index()
        m['model'], m['fold'] = model_name, fold
        records.append(m)
    print(f'{model_name}: {len(ckpts)} checkpoints done', flush=True)
usage = pd.concat(records, ignore_index=True)
usage.to_csv(EXPERIMENT/'diagnosis_fusion_usage.csv', index=False)
print('gate = softmax weight per piece; share = fraction of the fused vector\'s magnitude from that piece (what really flows on).')
print('In esm_mlp the two "field" pieces carry the mutation impulse, not structure; in no_structure the field has no 3D geometry.')
display(usage.groupby('model')[[c for c in usage if '|' in c]].mean().T.round(3))

# ---- C. by task type ---------------------------------------------------------------------------------------
print('\n=== C. BY TASK TYPE (test genes; assays within gene, genes equal, then mean over seeds) ===')
t = pd.read_csv(EXPERIMENT/'matched_comparison_all.csv')
per = t.groupby(['task', 'model', 'seed', 'gene'])[['model_spearman', 'llr_spearman']].mean() \
       .groupby(level=['task', 'model', 'seed']).mean()
task_tab = per.model_spearman.groupby(level=['task', 'model']).mean().unstack('model')
task_tab['LLR'] = per.llr_spearman.groupby(level='task').mean()
task_tab['genes'] = t.groupby('task').gene.nunique()
display(task_tab.round(3))
print('Seed SD per task/model:')
display(per.model_spearman.groupby(level=['task', 'model']).std().unstack('model').round(3))
task_tab.to_csv(EXPERIMENT/'diagnosis_by_task.csv')
print('\nPaste tables A, B and C back. Same caveat as before: 10 test genes, seed SD is not gene uncertainty.')


=== A. MECHANISM SEPARATION (two assays on the same variants, e.g. abundance vs activity) ===
pair status counts: {'ok': 132}


rank_contrast_spearman        mae_gain       
                                                  mean    std     mean    std
model            part                                                        
esm_mlp          mechanism_only                  0.053  0.010   -0.019  0.024
                 ndd_evaluation                 -0.027  0.066   -0.015  0.012
mipo             mechanism_only                  0.044  0.025   -0.003  0.006
                 ndd_evaluation                 -0.028  0.050   -0.010  0.008
no_global_kernel mechanism_only                  0.047  0.008   -0.005  0.005
                 ndd_evaluation                 -0.013  0.019   -0.011  0.007
no_structure     mechanism_only                  0.038  0.030   -0.013  0.018
                 ndd_evaluation                  0.033  0.035   -0.004  0.003

Abundance-anchored pairs only (stability proxy vs function):


,mean,std
model,,
esm_mlp,0.040,0.024
mipo,0.033,0.028
no_global_kernel,0.038,0.006
no_structure,0.037,0.035


Per gene (mean over pairs and seeds):


...


=== C. BY TASK TYPE (test genes; assays within gene, genes equal, then mean over seeds) ===


model,esm_mlp,mipo,no_global_kernel,no_structure,LLR,genes
task,,,,,,
abundance,0.440,0.447,0.449,0.447,0.417,9
activity,0.421,0.465,0.462,0.457,0.490,6
binding,0.488,0.482,0.475,0.477,0.523,3
fitness,0.479,0.490,0.486,0.490,0.494,2
stability,0.564,0.565,0.563,0.563,0.470,1


Seed SD per task/model:


model,esm_mlp,mipo,no_global_kernel,no_structure
task,,,,
abundance,0.006,0.007,0.005,0.015
activity,0.031,0.004,0.012,0.017
binding,0.009,0.005,0.009,0.009
fitness,0.002,0.012,0.021,0.008
stability,0.003,0.012,0.007,0.012



Paste tables A, B and C back. Same caveat as before: 10 test genes, seed SD is not gene uncertainty.
